<a href="https://colab.research.google.com/github/jananagaty/t19/blob/main/thesis_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch pandas numpy scikit-learn tqdm requests spotipy pykeen networkx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.4/404.4 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
import os

drive_dataset_path = "/content/drive/MyDrive/spotify dataset"

print(os.listdir(drive_dataset_path))

['spotify_million_playlist_dataset.zip']


In [6]:
zip_path = "/content/drive/MyDrive/spotify dataset/spotify_million_playlist_dataset.zip"

!unzip -q "$zip_path" -d /content/dataset

In [7]:
import os

dataset_path = "/content/dataset"

print(os.listdir(dataset_path)[:10])

['README.md', 'data', 'stats.txt', 'src', 'license.txt', 'md5sums']


In [8]:
import glob

json_files = glob.glob("/content/dataset/data/*.json")

print("Number of JSON files:", len(json_files))
print(json_files[:5])

Number of JSON files: 1000
['/content/dataset/data/mpd.slice.480000-480999.json', '/content/dataset/data/mpd.slice.367000-367999.json', '/content/dataset/data/mpd.slice.13000-13999.json', '/content/dataset/data/mpd.slice.135000-135999.json', '/content/dataset/data/mpd.slice.322000-322999.json']


In [9]:
import json

with open(json_files[0], "r") as f:
    data = json.load(f)

print(data.keys())

dict_keys(['info', 'playlists'])


In [10]:
playlist = data['playlists'][0]

print(playlist.keys())

dict_keys(['name', 'collaborative', 'pid', 'modified_at', 'num_tracks', 'num_albums', 'num_followers', 'tracks', 'num_edits', 'duration_ms', 'num_artists'])


In [9]:
import os
import glob
import json
import gc
import pandas as pd
from tqdm import tqdm

In [2]:
json_files = glob.glob("/content/dataset/**/*.json", recursive=True)
json_files = [f for f in json_files if "mpd.slice" in os.path.basename(f)]
json_files = sorted(json_files)

print("Number of MPD slice files found:", len(json_files))
print("First 5 files:", json_files[:5])

Number of MPD slice files found: 1000
First 5 files: ['/content/dataset/data/mpd.slice.0-999.json', '/content/dataset/data/mpd.slice.1000-1999.json', '/content/dataset/data/mpd.slice.10000-10999.json', '/content/dataset/data/mpd.slice.100000-100999.json', '/content/dataset/data/mpd.slice.101000-101999.json']


In [3]:
# Start small first to make sure everything works
json_files = json_files[:50]

print("Using", len(json_files), "JSON files for now")

Using 50 JSON files for now


In [4]:
output_folder = "/content/output/interim"
os.makedirs(output_folder, exist_ok=True)

output_csv = os.path.join(output_folder, "playlists_flat.csv")

# If file already exists from an old run, delete it first
if os.path.exists(output_csv):
    os.remove(output_csv)

first_write = True
total_rows = 0

for file_path in tqdm(json_files, desc="Flattening JSON files safely"):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    for playlist in data["playlists"]:
        playlist_id = playlist.get("pid")
        playlist_name = playlist.get("name")
        num_tracks = playlist.get("num_tracks")
        num_albums = playlist.get("num_albums")
        num_artists = playlist.get("num_artists")
        modified_at = playlist.get("modified_at")

        tracks = playlist.get("tracks", [])

        for position, track in enumerate(tracks):
            rows.append({
                "playlist_id": playlist_id,
                "playlist_name": playlist_name,
                "num_tracks_reported": num_tracks,
                "num_albums_reported": num_albums,
                "num_artists_reported": num_artists,
                "modified_at": modified_at,
                "track_position": position,
                "track_uri": track.get("track_uri"),
                "track_name": track.get("track_name"),
                "artist_uri": track.get("artist_uri"),
                "artist_name": track.get("artist_name"),
                "album_uri": track.get("album_uri"),
                "album_name": track.get("album_name"),
                "duration_ms": track.get("duration_ms")
            })

    # Convert only this file's rows to a dataframe
    chunk_df = pd.DataFrame(rows)
    total_rows += len(chunk_df)

    # Append to CSV
    chunk_df.to_csv(output_csv, mode="a", header=first_write, index=False)
    first_write = False

    # Free memory aggressively
    del data
    del rows
    del chunk_df
    gc.collect()

print("Flattening complete.")
print("Saved file:", output_csv)
print("Total rows written:", total_rows)

Flattening JSON files safely: 100%|██████████| 50/50 [00:53<00:00,  1.08s/it]

Flattening complete.
Saved file: /content/output/interim/playlists_flat.csv
Total rows written: 3348258


In [5]:
playlists_flat = pd.read_csv("/content/output/interim/playlists_flat.csv")

print("Shape:", playlists_flat.shape)
playlists_flat.head()

Shape: (3348258, 14)


,playlist_id,playlist_name,num_tracks_reported,num_albums_reported,num_artists_reported,modified_at,track_position,track_uri,track_name,artist_uri,artist_name,album_uri,album_name,duration_ms
0,0,Throwbacks,52,47,37,1493424000,0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,Lose Control (feat. Ciara & Fat Man Scoop),spotify:artist:2wIVse2owClT7go1WT98tk,Missy Elliott,spotify:album:6vV5UrXcfyQD1wu4Qo2I9K,The Cookbook,226863
1,0,Throwbacks,52,47,37,1493424000,1,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,Toxic,spotify:artist:26dSoYclwsYLMAKD3tpOr4,Britney Spears,spotify:album:0z7pVBGOD7HCIB7S8eLkLI,In The Zone,198800
2,0,Throwbacks,52,47,37,1493424000,2,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,Crazy In Love,spotify:artist:6vWDO969PvNqNYHIOW5v0m,Beyoncé,spotify:album:25hVFAxTlDvXbx2X2QkUkE,Dangerously In Love (Alben für die Ewigkeit),235933
3,0,Throwbacks,52,47,37,1493424000,3,spotify:track:1AWQoqb9bSvzTjaLralEkT,Rock Your Body,spotify:artist:31TPClRtHm23RisEBtV3X7,Justin Timberlake,spotify:album:6QPkyl04rXwTGlGlcYaRoW,Justified,267266
4,0,Throwbacks,52,47,37,1493424000,4,spotify:track:1lzr43nnXAijIGYnCT8M8H,It Wasn't Me,spotify:artist:5EvFsr3kj42KNv97ZEnqij,Shaggy,spotify:album:6NmFmPX56pcLBOFMhIiKvF,Hot Shot,227600


In [10]:
flat_csv_path = "/content/output/interim/playlists_flat.csv"

use_cols = [
    "playlist_id",
    "playlist_name",
    "track_position",
    "track_uri",
    "track_name",
    "artist_uri",
    "artist_name",
    "album_uri",
    "album_name",
    "duration_ms"
]

df = pd.read_csv(flat_csv_path, usecols=use_cols)

print("Loaded flattened CSV.")
print("Shape:", df.shape)
df.head()

Loaded flattened CSV.
Shape: (3348258, 10)


,playlist_id,playlist_name,track_position,track_uri,track_name,artist_uri,artist_name,album_uri,album_name,duration_ms
0,0,Throwbacks,0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,Lose Control (feat. Ciara & Fat Man Scoop),spotify:artist:2wIVse2owClT7go1WT98tk,Missy Elliott,spotify:album:6vV5UrXcfyQD1wu4Qo2I9K,The Cookbook,226863
1,0,Throwbacks,1,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,Toxic,spotify:artist:26dSoYclwsYLMAKD3tpOr4,Britney Spears,spotify:album:0z7pVBGOD7HCIB7S8eLkLI,In The Zone,198800
2,0,Throwbacks,2,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,Crazy In Love,spotify:artist:6vWDO969PvNqNYHIOW5v0m,Beyoncé,spotify:album:25hVFAxTlDvXbx2X2QkUkE,Dangerously In Love (Alben für die Ewigkeit),235933
3,0,Throwbacks,3,spotify:track:1AWQoqb9bSvzTjaLralEkT,Rock Your Body,spotify:artist:31TPClRtHm23RisEBtV3X7,Justin Timberlake,spotify:album:6QPkyl04rXwTGlGlcYaRoW,Justified,267266
4,0,Throwbacks,4,spotify:track:1lzr43nnXAijIGYnCT8M8H,It Wasn't Me,spotify:artist:5EvFsr3kj42KNv97ZEnqij,Shaggy,spotify:album:6NmFmPX56pcLBOFMhIiKvF,Hot Shot,227600


In [11]:
rows_before = len(df)

# Remove rows missing essential fields
df = df.dropna(subset=["playlist_id", "track_position", "track_uri", "artist_uri"]).copy()

# Convert to integer where needed
df["playlist_id"] = df["playlist_id"].astype(int)
df["track_position"] = df["track_position"].astype(int)

# Remove duplicate positions inside the same playlist
df = df.drop_duplicates(subset=["playlist_id", "track_position"], keep="first").copy()

# Sort correctly
df = df.sort_values(["playlist_id", "track_position"]).reset_index(drop=True)

rows_after = len(df)

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)
print("Rows removed:", rows_before - rows_after)
df.head()

Rows before cleaning: 3348258
Rows after cleaning: 3348258
Rows removed: 0


,playlist_id,playlist_name,track_position,track_uri,track_name,artist_uri,artist_name,album_uri,album_name,duration_ms
0,0,Throwbacks,0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,Lose Control (feat. Ciara & Fat Man Scoop),spotify:artist:2wIVse2owClT7go1WT98tk,Missy Elliott,spotify:album:6vV5UrXcfyQD1wu4Qo2I9K,The Cookbook,226863
1,0,Throwbacks,1,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,Toxic,spotify:artist:26dSoYclwsYLMAKD3tpOr4,Britney Spears,spotify:album:0z7pVBGOD7HCIB7S8eLkLI,In The Zone,198800
2,0,Throwbacks,2,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,Crazy In Love,spotify:artist:6vWDO969PvNqNYHIOW5v0m,Beyoncé,spotify:album:25hVFAxTlDvXbx2X2QkUkE,Dangerously In Love (Alben für die Ewigkeit),235933
3,0,Throwbacks,3,spotify:track:1AWQoqb9bSvzTjaLralEkT,Rock Your Body,spotify:artist:31TPClRtHm23RisEBtV3X7,Justin Timberlake,spotify:album:6QPkyl04rXwTGlGlcYaRoW,Justified,267266
4,0,Throwbacks,4,spotify:track:1lzr43nnXAijIGYnCT8M8H,It Wasn't Me,spotify:artist:5EvFsr3kj42KNv97ZEnqij,Shaggy,spotify:album:6NmFmPX56pcLBOFMhIiKvF,Hot Shot,227600


In [12]:
playlist_lengths = df.groupby("playlist_id").size().reset_index(name="sequence_length")

print("Number of playlists:", len(playlist_lengths))
print("Min length:", playlist_lengths["sequence_length"].min())
print("Max length:", playlist_lengths["sequence_length"].max())
playlist_lengths.head()

Number of playlists: 50000
Min length: 5
Max length: 250


,playlist_id,sequence_length
0,0,52
1,1,39
2,2,64
3,3,126
4,4,17


In [13]:
MIN_PLAYLIST_LENGTH = 20

valid_playlist_ids = playlist_lengths.loc[
    playlist_lengths["sequence_length"] >= MIN_PLAYLIST_LENGTH,
    "playlist_id"
].tolist()

df_filtered = df[df["playlist_id"].isin(valid_playlist_ids)].copy()

print("Playlists before filtering:", df["playlist_id"].nunique())
print("Playlists after filtering:", df_filtered["playlist_id"].nunique())
print("Rows after filtering:", len(df_filtered))

Playlists before filtering: 50000
Playlists after filtering: 42043
Rows after filtering: 3243624


In [18]:
# import numpy as np
# TARGET_NUM_USERS = 1006
# np.random.seed(42)
# available_ids = sorted(df_filtered["playlist_id"].unique())
# selected_playlist_ids = np.random.choice(
#     available_ids,
#     size=min(TARGET_NUM_USERS, len(available_ids)),
#     replace=False
# )
# df_selected = df_filtered[df_filtered["playlist_id"].isin(selected_playlist_ids)].copy()
# print("Selected playlists/users:", df_selected["playlist_id"].nunique())
# print("Selected rows:", len(df_selected))

In [19]:
del df
del df_filtered
del playlist_lengths
gc.collect()

383

In [20]:
unique_tracks = (
    df_selected[
        ["track_uri", "track_name", "artist_uri", "artist_name", "album_uri", "album_name"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Unique tracks:", len(unique_tracks))
unique_tracks.head()

Unique tracks: 38876


,track_uri,track_name,artist_uri,artist_name,album_uri,album_name
0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,Lose Control (feat. Ciara & Fat Man Scoop),spotify:artist:2wIVse2owClT7go1WT98tk,Missy Elliott,spotify:album:6vV5UrXcfyQD1wu4Qo2I9K,The Cookbook
1,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,Toxic,spotify:artist:26dSoYclwsYLMAKD3tpOr4,Britney Spears,spotify:album:0z7pVBGOD7HCIB7S8eLkLI,In The Zone
2,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,Crazy In Love,spotify:artist:6vWDO969PvNqNYHIOW5v0m,Beyoncé,spotify:album:25hVFAxTlDvXbx2X2QkUkE,Dangerously In Love (Alben für die Ewigkeit)
3,spotify:track:1AWQoqb9bSvzTjaLralEkT,Rock Your Body,spotify:artist:31TPClRtHm23RisEBtV3X7,Justin Timberlake,spotify:album:6QPkyl04rXwTGlGlcYaRoW,Justified
4,spotify:track:1lzr43nnXAijIGYnCT8M8H,It Wasn't Me,spotify:artist:5EvFsr3kj42KNv97ZEnqij,Shaggy,spotify:album:6NmFmPX56pcLBOFMhIiKvF,Hot Shot


In [21]:
unique_artists = (
    df_selected[["artist_uri", "artist_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Unique artists:", len(unique_artists))
unique_artists.head()

Unique artists: 10831


,artist_uri,artist_name
0,spotify:artist:2wIVse2owClT7go1WT98tk,Missy Elliott
1,spotify:artist:26dSoYclwsYLMAKD3tpOr4,Britney Spears
2,spotify:artist:6vWDO969PvNqNYHIOW5v0m,Beyoncé
3,spotify:artist:31TPClRtHm23RisEBtV3X7,Justin Timberlake
4,spotify:artist:5EvFsr3kj42KNv97ZEnqij,Shaggy


In [22]:
df_selected = df_selected.sort_values(["playlist_id", "track_position"]).reset_index(drop=True)

user_sequences = (
    df_selected.groupby("playlist_id")["track_uri"]
    .apply(list)
    .reset_index()
)

user_sequences.columns = ["user_id", "track_sequence"]
user_sequences["sequence_length"] = user_sequences["track_sequence"].apply(len)

user_sequences = user_sequences[["user_id", "sequence_length", "track_sequence"]]

print("Number of user sequences:", len(user_sequences))
user_sequences.head()

Number of user sequences: 1006


,user_id,sequence_length,track_sequence
0,0,52,"[spotify:track:0UaMYEvWZi0ZqiDOoHU3YI, spotify..."
1,1,39,"[spotify:track:2HHtWyy5CgaQbC7XSoOb0e, spotify..."
2,2,64,"[spotify:track:74tqql9zP6JjF5hjkHHUXp, spotify..."
3,3,126,"[spotify:track:4WJ7UMD4i6DOPzyXU5pZSz, spotify..."
4,5,80,"[spotify:track:61LtVmmkGr8P9I2tSPvdpf, spotify..."


In [23]:
user_sequences_to_save = user_sequences.copy()
user_sequences_to_save["track_sequence"] = user_sequences_to_save["track_sequence"].apply(json.dumps)

user_sequences_to_save.head()

,user_id,sequence_length,track_sequence
0,0,52,"[""spotify:track:0UaMYEvWZi0ZqiDOoHU3YI"", ""spot..."
1,1,39,"[""spotify:track:2HHtWyy5CgaQbC7XSoOb0e"", ""spot..."
2,2,64,"[""spotify:track:74tqql9zP6JjF5hjkHHUXp"", ""spot..."
3,3,126,"[""spotify:track:4WJ7UMD4i6DOPzyXU5pZSz"", ""spot..."
4,5,80,"[""spotify:track:61LtVmmkGr8P9I2tSPvdpf"", ""spot..."


In [24]:
def split_sequence(seq, train_ratio=0.9):
    split_index = int(len(seq) * train_ratio)

    # Prevent empty test set
    if split_index >= len(seq):
        split_index = len(seq) - 1

    train_seq = seq[:split_index]
    test_seq = seq[split_index:]

    return train_seq, test_seq


train_rows = []
test_rows = []

for _, row in user_sequences.iterrows():
    user_id = row["user_id"]
    seq = row["track_sequence"]

    train_seq, test_seq = split_sequence(seq, train_ratio=0.9)

    train_rows.append({
        "user_id": user_id,
        "train_length": len(train_seq),
        "train_sequence": json.dumps(train_seq)
    })

    test_rows.append({
        "user_id": user_id,
        "test_length": len(test_seq),
        "test_sequence": json.dumps(test_seq)
    })

train_sequences = pd.DataFrame(train_rows)
test_sequences = pd.DataFrame(test_rows)

print("Train sequences shape:", train_sequences.shape)
print("Test sequences shape:", test_sequences.shape)

train_sequences.head()

Train sequences shape: (1006, 3)
Test sequences shape: (1006, 3)


,user_id,train_length,train_sequence
0,0,46,"[""spotify:track:0UaMYEvWZi0ZqiDOoHU3YI"", ""spot..."
1,1,35,"[""spotify:track:2HHtWyy5CgaQbC7XSoOb0e"", ""spot..."
2,2,57,"[""spotify:track:74tqql9zP6JjF5hjkHHUXp"", ""spot..."
3,3,113,"[""spotify:track:4WJ7UMD4i6DOPzyXU5pZSz"", ""spot..."
4,5,72,"[""spotify:track:61LtVmmkGr8P9I2tSPvdpf"", ""spot..."


In [25]:
processed_folder = "/content/output/processed"
os.makedirs(processed_folder, exist_ok=True)

In [26]:
df_selected.to_csv(f"{processed_folder}/playlist_tracks_clean.csv", index=False)
unique_tracks.to_csv(f"{processed_folder}/unique_tracks.csv", index=False)
unique_artists.to_csv(f"{processed_folder}/unique_artists.csv", index=False)
user_sequences_to_save.to_csv(f"{processed_folder}/user_sequences.csv", index=False)
train_sequences.to_csv(f"{processed_folder}/train_sequences.csv", index=False)
test_sequences.to_csv(f"{processed_folder}/test_sequences.csv", index=False)

print("All processed files saved successfully.")

All processed files saved successfully.


In [27]:
report_lines = []

report_lines.append("PREPROCESSING REPORT")
report_lines.append("====================")
report_lines.append(f"Selected playlists/users: {df_selected['playlist_id'].nunique()}")
report_lines.append(f"Selected rows: {len(df_selected)}")
report_lines.append(f"Unique tracks: {len(unique_tracks)}")
report_lines.append(f"Unique artists: {len(unique_artists)}")
report_lines.append(f"User sequences: {len(user_sequences)}")
report_lines.append(f"Minimum playlist length used: {MIN_PLAYLIST_LENGTH}")
report_lines.append(f"Target users/playlists: {TARGET_NUM_USERS}")

report_text = "\n".join(report_lines)

print(report_text)

with open(f"{processed_folder}/preprocessing_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

print("Saved preprocessing report.")

PREPROCESSING REPORT
Selected playlists/users: 1006
Selected rows: 78644
Unique tracks: 38876
Unique artists: 10831
User sequences: 1006
Minimum playlist length used: 20
Target users/playlists: 1006
Saved preprocessing report.


In [28]:
print("Cleaned playlist-track table:")
display(df_selected.head())

print("\nUnique tracks:")
display(unique_tracks.head())

print("\nUnique artists:")
display(unique_artists.head())

print("\nUser sequences:")
display(user_sequences.head())

print("\nTrain sequences:")
display(train_sequences.head())

print("\nTest sequences:")
display(test_sequences.head())

Cleaned playlist-track table:


,playlist_id,playlist_name,track_position,track_uri,track_name,artist_uri,artist_name,album_uri,album_name,duration_ms
0,0,Throwbacks,0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,Lose Control (feat. Ciara & Fat Man Scoop),spotify:artist:2wIVse2owClT7go1WT98tk,Missy Elliott,spotify:album:6vV5UrXcfyQD1wu4Qo2I9K,The Cookbook,226863
1,0,Throwbacks,1,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,Toxic,spotify:artist:26dSoYclwsYLMAKD3tpOr4,Britney Spears,spotify:album:0z7pVBGOD7HCIB7S8eLkLI,In The Zone,198800
2,0,Throwbacks,2,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,Crazy In Love,spotify:artist:6vWDO969PvNqNYHIOW5v0m,Beyoncé,spotify:album:25hVFAxTlDvXbx2X2QkUkE,Dangerously In Love (Alben für die Ewigkeit),235933
3,0,Throwbacks,3,spotify:track:1AWQoqb9bSvzTjaLralEkT,Rock Your Body,spotify:artist:31TPClRtHm23RisEBtV3X7,Justin Timberlake,spotify:album:6QPkyl04rXwTGlGlcYaRoW,Justified,267266
4,0,Throwbacks,4,spotify:track:1lzr43nnXAijIGYnCT8M8H,It Wasn't Me,spotify:artist:5EvFsr3kj42KNv97ZEnqij,Shaggy,spotify:album:6NmFmPX56pcLBOFMhIiKvF,Hot Shot,227600



Unique tracks:


,track_uri,track_name,artist_uri,artist_name,album_uri,album_name
0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,Lose Control (feat. Ciara & Fat Man Scoop),spotify:artist:2wIVse2owClT7go1WT98tk,Missy Elliott,spotify:album:6vV5UrXcfyQD1wu4Qo2I9K,The Cookbook
1,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,Toxic,spotify:artist:26dSoYclwsYLMAKD3tpOr4,Britney Spears,spotify:album:0z7pVBGOD7HCIB7S8eLkLI,In The Zone
2,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,Crazy In Love,spotify:artist:6vWDO969PvNqNYHIOW5v0m,Beyoncé,spotify:album:25hVFAxTlDvXbx2X2QkUkE,Dangerously In Love (Alben für die Ewigkeit)
3,spotify:track:1AWQoqb9bSvzTjaLralEkT,Rock Your Body,spotify:artist:31TPClRtHm23RisEBtV3X7,Justin Timberlake,spotify:album:6QPkyl04rXwTGlGlcYaRoW,Justified
4,spotify:track:1lzr43nnXAijIGYnCT8M8H,It Wasn't Me,spotify:artist:5EvFsr3kj42KNv97ZEnqij,Shaggy,spotify:album:6NmFmPX56pcLBOFMhIiKvF,Hot Shot



Unique artists:


...

,user_id,sequence_length,track_sequence
0,0,52,"[spotify:track:0UaMYEvWZi0ZqiDOoHU3YI, spotify..."
1,1,39,"[spotify:track:2HHtWyy5CgaQbC7XSoOb0e, spotify..."
2,2,64,"[spotify:track:74tqql9zP6JjF5hjkHHUXp, spotify..."
3,3,126,"[spotify:track:4WJ7UMD4i6DOPzyXU5pZSz, spotify..."
4,5,80,"[spotify:track:61LtVmmkGr8P9I2tSPvdpf, spotify..."



Train sequences:


,user_id,train_length,train_sequence
0,0,46,"[""spotify:track:0UaMYEvWZi0ZqiDOoHU3YI"", ""spot..."
1,1,35,"[""spotify:track:2HHtWyy5CgaQbC7XSoOb0e"", ""spot..."
2,2,57,"[""spotify:track:74tqql9zP6JjF5hjkHHUXp"", ""spot..."
3,3,113,"[""spotify:track:4WJ7UMD4i6DOPzyXU5pZSz"", ""spot..."
4,5,72,"[""spotify:track:61LtVmmkGr8P9I2tSPvdpf"", ""spot..."



Test sequences:


,user_id,test_length,test_sequence
0,0,6,"[""spotify:track:3utIAb67sOu0QHxBE88P1M"", ""spot..."
1,1,4,"[""spotify:track:0K6yUnIKNsFtfIpTgGtcHm"", ""spot..."
2,2,7,"[""spotify:track:5NB0u5gWcwH1D6y3euVSKI"", ""spot..."
3,3,13,"[""spotify:track:2A6arq0CZXZZiwL4Wp6J2Y"", ""spot..."
4,5,8,"[""spotify:track:6FE2iI43OZnszFLuLtvvmg"", ""spot..."


In [29]:
from google.colab import files

files.download(f"{processed_folder}/unique_tracks.csv")
files.download(f"{processed_folder}/unique_artists.csv")
files.download(f"{processed_folder}/user_sequences.csv")
files.download(f"{processed_folder}/train_sequences.csv")
files.download(f"{processed_folder}/test_sequences.csv")
files.download(f"{processed_folder}/preprocessing_report.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>